# Project: Pose Tracking for Better Stance during Billiards Play

Dev TODOS
- [x] Set up base structure
  - [x] Installed packages
  - [x] Test sample loop with live video
    - [ ] ~~(Not a) Bug: The capturing window does not close when ESC is tapped~~
- [ ] Bug(DX): Pylance does not detect attributes from imported packages
- [ ] Feature: Able to list available video sources then select one
- [x] Enhancement: Draw only necessary pose landmarks
- [ ] Study: Define optimal pose during exercise
  - [ ] Feature: Store the pose configuration
  - [ ] Feature: Calculate angles from tracked landmarks
  - [ ] 

Product TODOS
- [ ] Define User Journey when using the app during training session
  - [ ] Setup of Training space/area

**References**

- https://youtu.be/pG4sUNDOZFg?si=qBYEat5cQ_c5tyhT
- https://learnopencv.com/introduction-to-mediapipe/
- https://github.com/spmallick/learnopencv/tree/master

In [1]:
# Load up the necessary libraries
import cv2
import mediapipe as mp

In [2]:
mp_pose = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

#### 1. Get video feed

In [5]:
# cap = cv2.VideoCapture(0) # connected ios
cap = cv2.VideoCapture(2) # default webcam
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow('Raw Webcam Feed', frame)
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [6]:
# Re-release from notebook may not work in MacOS
cap.release()
cv2.destroyAllWindows()

#### 1.5. Configure Styling

In [29]:
mp_drawing.DrawingSpec??

Init signature:
mp_drawing.DrawingSpec(
    color: Tuple[int, int, int] = (224, 224, 224),
    thickness: int = 2,
    circle_radius: int = 2,
) -> None
Docstring:      DrawingSpec(color: Tuple[int, int, int] = (224, 224, 224), thickness: int = 2, circle_radius: int = 2)
Source:        
@dataclasses.dataclass
class DrawingSpec:
  # Color for drawing the annotation. Default to the white color.
  color: Tuple[int, int, int] = WHITE_COLOR
  # Thickness for drawing the annotation. Default to 2 pixels.
  thickness: int = 2
  # Circle radius. Default to 2 pixels.
  circle_radius: int = 2
File:           ~/CodeSpace/my_workshop/game_billiards/study/.venv/lib/python3.12/site-packages/mediapipe/python/solutions/drawing_utils.py
Type:           type
Subclasses:     

In [31]:
mp_drawing.draw_landmarks??

Signature:
mp_drawing.draw_landmarks(
    image: numpy.ndarray,
    landmark_list: mediapipe.framework.formats.landmark_pb2.NormalizedLandmarkList,
    connections: Optional[List[Tuple[int, int]]] = None,
    landmark_drawing_spec: Union[mediapipe.python.solutions.drawing_utils.DrawingSpec, Mapping[int, mediapipe.python.solutions.drawing_utils.DrawingSpec], NoneType] = DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2),
    connection_drawing_spec: Union[mediapipe.python.solutions.drawing_utils.DrawingSpec, Mapping[Tuple[int, int], mediapipe.python.solutions.drawing_utils.DrawingSpec]] = DrawingSpec(color=(224, 224, 224), thickness=2, circle_radius=2),
    is_drawing_landmarks: bool = True,
)
Source:   
def draw_landmarks(
    image: np.ndarray,
    landmark_list: landmark_pb2.NormalizedLandmarkList,
    connections: Optional[List[Tuple[int, int]]] = None,
    landmark_drawing_spec: Optional[
        Union[DrawingSpec, Mapping[int, DrawingSpec]]
    ] = DrawingSpec(color=RED_

#### 2. Make Detections from Feed

In [30]:
# Check availabe pose landmarks & connections
mp_pose.PoseLandmark??

Init signature: mp_pose.PoseLandmark(*values)
Source:        
class PoseLandmark(enum.IntEnum):
  """The 33 pose landmarks."""
  NOSE = 0
  LEFT_EYE_INNER = 1
  LEFT_EYE = 2
  LEFT_EYE_OUTER = 3
  RIGHT_EYE_INNER = 4
  RIGHT_EYE = 5
  RIGHT_EYE_OUTER = 6
  LEFT_EAR = 7
  RIGHT_EAR = 8
  MOUTH_LEFT = 9
  MOUTH_RIGHT = 10
  LEFT_SHOULDER = 11
  RIGHT_SHOULDER = 12
  LEFT_ELBOW = 13
  RIGHT_ELBOW = 14
  LEFT_WRIST = 15
  RIGHT_WRIST = 16
  LEFT_PINKY = 17
  RIGHT_PINKY = 18
  LEFT_INDEX = 19
  RIGHT_INDEX = 20
  LEFT_THUMB = 21
  RIGHT_THUMB = 22
  LEFT_HIP = 23
  RIGHT_HIP = 24
  LEFT_KNEE = 25
  RIGHT_KNEE = 26
  LEFT_ANKLE = 27
  RIGHT_ANKLE = 28
  LEFT_HEEL = 29
  RIGHT_HEEL = 30
  LEFT_FOOT_INDEX = 31
  RIGHT_FOOT_INDEX = 32
File:           ~/CodeSpace/my_workshop/game_billiards/study/.venv/lib/python3.12/site-packages/mediapipe/python/solutions/pose.py
Type:           EnumType
Subclasses:     

In [10]:
# cap = cv2.VideoCapture(0) # connected ios
cap = cv2.VideoCapture(2) # default webcam

# Initiate holistic model
with mp_pose.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        # Capture feed by frame, and recoloring for better suitability for the model
        ret, frame = cap.read()
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Make Detections
        results = holistic.process(image)
        # print(results.face_landmarks)
        # print(results.left_hand_landmarks)
        # print(results.rigt_hand_landmarks)
        # print(results.pose_landmarks)
        
        # Recolor image back to BGR for rendering
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Draw face landmarks; NOTE: may no need
        # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_pose.FACEMESH_CONTOURS)
        # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_pose.FACEMESH_TESSELATION,
        #                           mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
        #                           mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))

        # Pose Detections (with a little face mesh)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        # NOTE: some landmarks may overlap with hand mesh
        # Right & Left hand
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_pose.HAND_CONNECTIONS,
                                  landmark_drawing_spec=mp_drawing.DrawingSpec(color=(59,92,100), thickness=2, circle_radius=2),
                                  connection_drawing_spec=mp_drawing.DrawingSpec(color=(213,0,55), thickness=2, circle_radius=2))
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_pose.HAND_CONNECTIONS)
                        
        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1762345243.720615 13643946 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1762345243.830335 13665519 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762345243.847668 13665524 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762345243.850313 13665522 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762345243.850548 13665517 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762345243.851747 13665524 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling su